# SwingTrader — 00 Setup Verification

**Run every cell top to bottom. All must pass before opening `01_data_layer.ipynb`.**

This notebook costs nothing — no LLM calls, no Kite tokens consumed.

| Check | What it verifies |
|---|---|
| Libraries | All packages installed at correct versions |
| `.env` | OPENAI_API_KEY present |
| OpenAI | Key is valid — returns `OK` |
| yfinance | Can fetch NSE data |
| Kite | Access token works (skip if not yet obtained) |

In [1]:
# Cell 1 — Install / upgrade all dependencies
# Run once; comment out after first successful run
%pip install -q yfinance pandas-ta langchain-openai langgraph langsmith \
    pydantic python-dotenv plotly scipy ipywidgets kiteconnect --upgrade

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Library versions
import importlib, sys
import importlib.metadata

def _get_version(pkg_import, pkg_dist=None):
    """Get version via __version__, .version, or importlib.metadata fallback."""
    mod = importlib.import_module(pkg_import)
    for attr in ('__version__', 'version', 'VERSION'):
        v = getattr(mod, attr, None)
        if v and isinstance(v, str):
            return v
    # fallback to dist metadata (e.g. pandas-ta vs pandas_ta)
    try:
        return importlib.metadata.version(pkg_dist or pkg_import)
    except importlib.metadata.PackageNotFoundError:
        return 'unknown'

checks = {
    "python":           sys.version.split()[0],
    "yfinance":         _get_version('yfinance'),
    "pandas_ta":        _get_version('pandas_ta', 'pandas-ta'),
    "langchain_openai": _get_version('langchain_openai'),
    "langgraph":        _get_version('langgraph'),
    "pydantic":         _get_version('pydantic'),
    "plotly":           _get_version('plotly'),
}

print('Library versions:')
for lib, ver in checks.items():
    print(f'  {lib:<22} {ver}')
print()
assert sys.version_info >= (3, 11), 'Need Python 3.11+'
print('✓ Python version OK')

Library versions:
  python                 3.12.13
  yfinance               1.4.1
  pandas_ta              0.4.71b0
  langchain_openai       1.3.0
  langgraph              1.2.4
  pydantic               2.13.4
  plotly                 6.8.0

✓ Python version OK


In [3]:
# Cell 3 — Load .env and verify OPENAI_API_KEY
from dotenv import load_dotenv
from pathlib import Path
import os

# Search up to 4 parent directories for a .env file
_here = Path.cwd()
_env_path = next(
    (p / '.env' for p in [_here, *_here.parents[:3]] if (p / '.env').exists()),
    None
)
if _env_path is None:
    raise FileNotFoundError(
        f'No .env file found in {_here} or its parents. '
        'Create one with OPENAI_API_KEY=sk-...'
    )
load_dotenv(dotenv_path=_env_path, override=True)
print(f'✓ .env loaded from {_env_path}')

api_key = os.getenv('OPENAI_API_KEY', '')
assert api_key.startswith('sk-'), \
    'OPENAI_API_KEY missing or malformed. Add to .env file: OPENAI_API_KEY=sk-...'
print(f'✓ OPENAI_API_KEY loaded  (ends: ...{api_key[-4:]})')

ls_key = os.getenv('LANGCHAIN_API_KEY', '')
if ls_key:
    print(f'✓ LANGCHAIN_API_KEY loaded (LangSmith tracing active)')
else:
    print('⚠  LANGCHAIN_API_KEY not set — LangSmith tracing disabled (optional but recommended)')

✓ .env loaded from /Users/shilpa/AI/SwingTrader/.env
✓ OPENAI_API_KEY loaded  (ends: ...YpsA)
✓ LANGCHAIN_API_KEY loaded (LangSmith tracing active)


In [4]:
# Cell 4 — OpenAI smoke test  (~$0.0001 cost)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)
response = llm.invoke('Reply with the single word: OK')

assert 'OK' in response.content, f'Unexpected response: {response.content}'
print(f'✓ OpenAI connection OK  (response: "{response.content.strip()}")')

✓ OpenAI connection OK  (response: "OK")


In [5]:
# Cell 5 — yfinance smoke test
import yfinance as yf
import pandas as pd

df = yf.download('INFY.NS', period='5d', auto_adjust=True, progress=False)

# Flatten MultiIndex if present (yfinance quirk)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

assert len(df) > 0, 'yfinance returned empty DataFrame — check internet connection'
assert 'Close' in df.columns or 'close' in df.columns, 'Unexpected column names'

print(f'✓ yfinance OK  ({len(df)} rows for INFY.NS, last date: {df.index[-1].date()})')
df.tail(2)

✓ yfinance OK  (5 rows for INFY.NS, last date: 2026-06-11)


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-06-10,1145.300049,1171.400024,1143.300049,1160.099976,8887626
2026-06-11,1119.300049,1133.699951,1109.000000,1129.699951,3905697


In [8]:
# Cell 6 — Kite Connect smoke test
# Run Cell 7 first if KITE_ACCESS_TOKEN is not yet set

KITE_API_KEY      = os.getenv('KITE_API_KEY', '')
KITE_ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN', '')

if not KITE_API_KEY:
    print('⚠  KITE_API_KEY not set — skipping Kite test')
    print('   Add to .env: KITE_API_KEY=your_key')
    print('   Evening 1 will use yfinance instead of Kite')
elif not KITE_ACCESS_TOKEN:
    print('⚠  KITE_ACCESS_TOKEN not set — run Cell 7 to generate it automatically')
else:
    from kiteconnect import KiteConnect
    kite = KiteConnect(api_key=KITE_API_KEY)
    kite.set_access_token(KITE_ACCESS_TOKEN)
    profile = kite.profile()
    print(f'✓ Kite Connect OK  (user: {profile["user_name"]})')

✓ Kite Connect OK  (user: Arnab Mohanty)


In [9]:
# Cell 7 — Generate Kite access token via local OAuth callback server
#
# Prerequisites:
#   • KITE_API_KEY and KITE_SECRET must be set in .env
#   • Redirect URL in Kite developer console must be: http://localhost:8000/kite/callback
#
# Run this cell once per day (token expires at ~08:00 IST).
# Skips automatically if a valid token is already loaded in this session.

import threading
import webbrowser
import socket as _socket
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs
from pathlib import Path
from kiteconnect import KiteConnect
from dotenv import set_key

KITE_API_KEY      = os.getenv('KITE_API_KEY', '')
KITE_SECRET       = os.getenv('KITE_SECRET', '')
KITE_ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN', '')

if not KITE_API_KEY or not KITE_SECRET:
    print('✗ KITE_API_KEY or KITE_SECRET not set in .env — skipping')
elif KITE_ACCESS_TOKEN:
    print('✓ KITE_ACCESS_TOKEN already set — skipping (re-run Cell 6 to verify)')
    print('  To refresh the token, clear KITE_ACCESS_TOKEN from .env and re-run this cell.')
else:
    _token_result = {}

    class _ReuseHTTPServer(HTTPServer):
        allow_reuse_address = True  # prevents OSError if port still held from previous run

    class _CallbackHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            parsed = urlparse(self.path)
            if parsed.path == '/kite/callback':
                params = parse_qs(parsed.query)
                request_token = params.get('request_token', [None])[0]
                status = params.get('status', ['unknown'])[0]

                if request_token and status == 'success':
                    try:
                        kite = KiteConnect(api_key=KITE_API_KEY)
                        data = kite.generate_session(request_token, api_secret=KITE_SECRET)
                        access_token = data['access_token']

                        env_file = next(
                            (p / '.env' for p in [Path.cwd(), *Path.cwd().parents[:3]] if (p / '.env').exists()),
                            Path.cwd() / '.env'
                        )
                        set_key(str(env_file), 'KITE_ACCESS_TOKEN', access_token)
                        os.environ['KITE_ACCESS_TOKEN'] = access_token

                        _token_result['access_token'] = access_token
                        body = b'<h2>Login successful. You can close this tab.</h2>'
                        self.send_response(200)
                    except Exception as e:
                        _token_result['error'] = str(e)
                        body = f'<h2>Error: {e}</h2>'.encode()
                        self.send_response(500)
                else:
                    _token_result['error'] = f'Login failed or cancelled (status={status})'
                    body = b'<h2>Login failed or cancelled. Try again.</h2>'
                    self.send_response(400)

                self.send_header('Content-Type', 'text/html')
                self.end_headers()
                self.wfile.write(body)
            else:
                self.send_response(404)
                self.end_headers()

            threading.Thread(target=self.server.shutdown, daemon=True).start()

        def log_message(self, *args):
            pass  # suppress access logs

    server = _ReuseHTTPServer(('localhost', 8000), _CallbackHandler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()

    login_url = f'https://kite.zerodha.com/connect/login?api_key={KITE_API_KEY}&v=3'
    print('Opening Kite login URL...')
    print(f'  {login_url}')
    webbrowser.open(login_url)
    print('Waiting for callback on http://localhost:8000/kite/callback ...')

    thread.join(timeout=120)

    if 'access_token' in _token_result:
        token = _token_result['access_token']
        print(f'✓ access_token saved to .env  (ends: ...{token[-4:]})')
    else:
        print(f'✗ Failed: {_token_result.get("error", "timeout — no callback received")}')

OSError: [Errno 48] Address already in use

## Summary

If all ✓ above:
- Open `01_data_layer.ipynb` → Evening 1 starts here

If ⚠ on Kite:
- Evening 1 will use **yfinance** instead — fully functional for research
- Kite is needed only when you move to paper trading (Phase 7)

If ✗ on OpenAI:
- Check `.env` file — key must start with `sk-`
- Verify key at https://platform.openai.com/api-keys